# Existing single-pose public continuity receiving
This separately registered population consumes the actual returned result from attempt 02, without a new search. The two-pose population remains incomplete. See e1_public_composition_2026-11-06.md.


In [ ]:
import base64
import gzip
import hashlib
import importlib.metadata as metadata
import json
import os
from copy import deepcopy
from datetime import datetime, timezone
from functools import wraps
from pathlib import Path

import molsysmt as msm
import numpy as np
import pyunitwizard as puw
import vina
from smonitor.integrations import CatalogException

import dockingmt as dmt

root = Path(os.environ['DOCKINGMT_E1_ROOT'])
output = Path(os.environ['DOCKINGMT_E1_COMPOSITION_OUTPUT'])
output.mkdir(exist_ok=False)
installed_check = globals()['registered_installed_check']
evidence = root / 'devguide/validation/data/e1_complete_pose_2026-11-06'
expected_root = '45b3665ba09b6979aa1bcdf1bea0ac3e8102f081fefbfc51348c55bc71405df3'
assert (
    hashlib.sha256((evidence / 'manifest.json').read_bytes()).hexdigest()
    == expected_root
)
for row in json.loads((evidence / 'manifest.json').read_text())['files']:
    data = (evidence / row['path']).read_bytes()
    assert (
        len(data) == row['bytes'] and hashlib.sha256(data).hexdigest() == row['sha256']
    )
    if row['path'].endswith('.gz'):
        decoded = gzip.decompress(data)
        assert (
            len(decoded) == row['decoded_bytes']
            and hashlib.sha256(decoded).hexdigest() == row['decoded_sha256']
        )
inputs = output / 'inputs'
inputs.mkdir()
source_path = inputs / 'full18.h5msm'
source_path.write_bytes(
    gzip.decompress(
        (evidence / 'normal/pose_1/full18_chemical_only.h5msm.gz').read_bytes()
    )
)
receptor_path = inputs / 'receptor.pdbqt'
receptor_path.write_bytes(
    gzip.decompress((evidence / 'provided_receptor.pdbqt.gz').read_bytes())
)
contract = json.loads((evidence / 'prospective_scoring_contract.json').read_text())
assert (
    hashlib.sha256(receptor_path.read_bytes()).hexdigest()
    == contract['receptor_sha256']
)
assert vina.__version__ == metadata.version('vina') == '1.2.7'
installed_check('before-scientific-work')

overlay = Path(os.environ['DOCKINGMT_E1_VINA_OVERLAY'])
shared = Path('/fast_data/home/liliana/miniconda3/envs/molsyssuite@uibcdf_3.14')


def digest_file(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def engine_payload():
    files = {}
    for record_path in (overlay / 'conda-meta').glob('*.json'):
        record = json.loads(record_path.read_text())
        for row in record['paths_data']['paths']:
            path = overlay / row['_path']
            assert path.resolve().is_relative_to(overlay.resolve())
            assert digest_file(path) == row['sha256']
            files[row['_path']] = row['sha256']
    assert len(files) == 103
    return files


def native_libraries():
    paths = set()
    for line in Path('/proc/self/maps').read_text().splitlines():
        parts = line.split(None, 5)
        if len(parts) == 6 and parts[5].startswith('/') and '.so' in parts[5]:
            path = Path(parts[5])
            if path.is_file():
                paths.add(path.resolve())
    return [dict(path=str(path), sha256=digest_file(path)) for path in sorted(paths)]


engine_before = engine_payload()
shared_metadata_before = {
    p.name: digest_file(p) for p in (shared / 'conda-meta').glob('*.json')
}
assert Path(vina.__file__).resolve().is_relative_to(overlay.resolve())
initial_libraries = native_libraries()
for name in ['libboost_filesystem.so.1.86.0', 'libboost_thread.so.1.86.0']:
    assert any(
        Path(row['path']).is_relative_to(overlay.resolve()) and name in row['path']
        for row in initial_libraries
    )

budgets = dict(
    prepare=2,
    reconstruct=1,
    dock_attempts=0,
    dock_returns=0,
    score_attempts=3,
    score_returns=2,
    graph_witness=4,
    charge_audit=2,
    expected_score_refusals=1,
    rank_refusals=0,
)
counts = dict.fromkeys(budgets, 0)
preparations = []
reconstructions = []
returned_scores = []


def plain(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if puw.is_quantity(value):
        return puw.convert(value, to_form='record').to_dict()
    raise TypeError(type(value).__name__)


def normalized(value):
    return json.loads(json.dumps(value, default=plain, allow_nan=False))


def save(path, value):
    with path.open('x') as stream:
        json.dump(value, stream, indent=2, allow_nan=False, default=plain)
        stream.write('\n')


def event(phase, **fields):
    with (output / 'operations.jsonl').open('a') as stream:
        stream.write(
            json.dumps(
                dict(
                    phase=phase, at_utc=datetime.now(timezone.utc).isoformat(), **fields
                ),
                allow_nan=False,
            )
            + '\n'
        )


def consume(name):
    assert counts[name] < budgets[name], name
    counts[name] += 1


def inspect_native(system, mechanics=True):
    fields = dict(
        ids=list(msm.get(system, element='atom', atom_id=True)),
        names=list(msm.get(system, element='atom', name=True)),
        coordinates_nm=puw.get_value(
            msm.get(system, element='atom', coordinates=True),
            to_unit='nm',
            value_type=list,
            dtype=float,
        ),
    )
    if mechanics:
        # This pinned native MolSysMT field returns bare values in canonical e.
        charge = puw.quantity(
            msm.get(system, element='atom', partial_charge=True), 'elementary_charge'
        )
        fields.update(
            charges_e=puw.get_value(
                charge, to_unit='elementary_charge', value_type=list, dtype=float
            ),
            charge_record=puw.convert(charge, to_form='record').to_dict(),
            types=list(msm.get(system, element='atom', atom_ff_type=True)),
            charge_assignment=system.molecular_mechanics.partial_charge_assignment,
            type_assignment=system.molecular_mechanics.atom_type_assignment,
        )
    return normalized(fields)


def inspect_prepared(prepared):
    return normalized(
        dict(
            declaration=prepared.to_dict(),
            native=inspect_native(prepared.source_molsys),
            coordinates_nm=puw.get_value(
                prepared.coordinates, to_unit='nm', value_type=list, dtype=float
            ),
            pdbqt=prepared.to_pdbqt(),
        )
    )


def export_fields(prepared):
    text = prepared.to_pdbqt()
    native = msm.convert(
        'pdbqt_text:' + text, to_form='molsysmt.MolSys', discard_torsion_tree=True
    )
    return normalized(
        dict(
            sha256=hashlib.sha256(text.encode()).hexdigest(),
            names=list(msm.get(native, element='atom', name=True)),
            coordinates_nm=puw.get_value(
                msm.get(native, element='atom', coordinates=True),
                to_unit='nm',
                value_type=list,
                dtype=float,
            ),
            charges_e=puw.get_value(
                puw.quantity(
                    msm.get(native, element='atom', partial_charge=True),
                    'elementary_charge',
                ),
                to_unit='elementary_charge',
                value_type=list,
                dtype=float,
            ),
            types=list(msm.get(native, element='atom', atom_ff_type=True)),
            tree=msm.form.string_pdbqt_text.get_torsion_tree('pdbqt_text:' + text),
        )
    )


def witness(system, reference, name):
    consume('graph_witness')
    left = inspect_native(system, mechanics=False)
    right = inspect_native(reference, mechanics=False)
    assert left['ids'] == right['ids'] and left['names'] == right['names'], (
        'Unexpected native axis; identity candidate not admitted'
    )
    n_atoms = len(left['ids'])
    report = msm.chemical_states.get_atom_correspondences(
        system,
        reference_molecular_system=reference,
        structure_indices=0,
        reference_structure_indices=0,
        chemical_state='structure',
        reference_chemical_state='structure',
        candidate_atom_correspondences=[[[i, i] for i in range(n_atoms)]],
        max_matches=1,
    )
    save(output / (name + '.json'), report)
    assert report['search_complete'] is True and report['n_candidate_matches'] == 1
    assert np.asarray(report['atom_correspondences']).shape == (1, n_atoms, 2)
    if n_atoms == 13:
        assert (
            report['source_graph_digest']
            == report['reference_graph_digest']
            == '9a5cbc3e84f0c5a2a65b66042c924635496ecda20582f811b86a5df92de7c5ba'
        )
    if n_atoms == 13:
        assert (
            report['source_graph_digest']
            == report['reference_graph_digest']
            == '9a5cbc3e84f0c5a2a65b66042c924635496ecda20582f811b86a5df92de7c5ba'
        )
    return normalized(report)


def charge_audit(prepared, name):
    consume('charge_audit')
    report = dmt.audit_preparation_charges(prepared)
    save(output / (name + '.json'), report)
    assert report['assessment'] == 'consistent'


def choose_cut(system):
    pair = msm.select(system, selection="atom_name in ['C3','C1']")
    assert len(pair) == 2
    return [[int(i) for i in pair]]


original_prepare = dmt.prepare_ligand
original_reconstruct = dmt.DockingPose.to_molecular_system
original_dock = dmt.dock
original_score = dmt.score


@wraps(original_prepare)
def monitored_prepare(*args, **kwargs):
    consume('prepare')
    event('prepare_started', attempt=counts['prepare'])
    prepared = original_prepare(*args, **kwargs)
    preparations.append(prepared)
    save(output / f'preparation_{counts["prepare"]}.json', inspect_prepared(prepared))
    event('prepare_returned', attempt=counts['prepare'])
    return prepared


@wraps(original_reconstruct)
def monitored_reconstruct(*args, **kwargs):
    consume('reconstruct')
    native = original_reconstruct(*args, **kwargs)
    reconstructions.append((native, inspect_native(native)))
    save(
        output / f'reconstruction_{counts["reconstruct"]}.json', reconstructions[-1][1]
    )
    return native


@wraps(original_dock)
def monitored_dock(*args, **kwargs):
    consume('dock_attempts')
    event('dock_started', attempt=counts['dock_attempts'])
    result = original_dock(*args, **kwargs)
    consume('dock_returns')
    save(output / 'source_result.json', result.to_dict())
    event('dock_returned', poses=len(result))
    return result


@wraps(original_score)
def monitored_score(*args, **kwargs):
    consume('score_attempts')
    event(
        'score_started', attempt=counts['score_attempts'], name=kwargs.get('score_name')
    )
    result = original_score(*args, **kwargs)
    counts['score_returns'] += 1
    save(output / f'returned_score_{counts["score_returns"]}.json', result.to_dict())
    assert counts['score_returns'] <= budgets['score_returns'], (
        'Unexpected extra scoring return preserved'
    )
    returned_scores.append(result)
    event(
        'score_returned',
        attempt=counts['score_attempts'],
        name=kwargs.get('score_name'),
    )
    return result


dmt.prepare_ligand = monitored_prepare
dmt.DockingPose.to_molecular_system = monitored_reconstruct
dmt.dock = monitored_dock
dmt.score = monitored_score

In [ ]:
caller_policy = puw.configure.report()
unit_context = puw.context(
    standard_units=[
        'pm',
        'fs',
        'kelvin',
        'mole',
        'dalton',
        'coulomb',
        'kJ/mol',
        'kJ/(mol*pm)',
        'kJ/(mol*pm**2)',
        'radians',
    ]
)
unit_context.__enter__()
active_policy = puw.configure.report()
full18 = msm.convert(source_path, to_form='molsysmt.MolSys')
full18_before = inspect_native(full18, mechanics=False)
assert len(full18_before['ids']) == 18
witness(full18, full18, 'full18_witness')
source_prepared = dmt.prepare_ligand(
    full18,
    selection='all',
    state_id='e1-public-composition-benzamidinium-20261106',
    active_torsion_bonds=choose_cut(full18),
    charge_options=dict(method='gasteiger_marsili'),
    typing_options=dict(typing_scheme='autodock4', method='chemical_environment'),
)
assert source_prepared.n_atoms == 13 and source_prepared.torsion_dof == 1
source_before = inspect_prepared(source_prepared)
assert source_prepared.pdbqt_atom_indices == [6, 7, 8, 9, 10, 11, 12, 0, 1, 2, 3, 4, 5]
charge_audit(source_prepared, 'source_charge_audit')
source_export = export_fields(source_prepared)
previous_attempt = (
    root / '.molsyssuite/e1-public-composition-20261106/attempt-02/native'
)
previous_bytes = (previous_attempt / 'source_result.json').read_bytes()
assert (
    hashlib.sha256(previous_bytes).hexdigest()
    == os.environ['DOCKINGMT_E1_EXISTING_RESULT_SHA256']
    == 'cd0e7e20a64034a5b1462f2c42810a83a0b247a4d1af56c237ceca13719fe55c'
)
previous_record = json.loads(previous_bytes)
source_result = dmt.DockingResult.from_dict(previous_record)
assert source_result.to_dict() == previous_record
save(output / 'source_result.json', previous_record)
original_capture = previous_record['provenance']['backend_artifacts']
dmt.verify_captured_inputs(original_capture)
assert (
    hashlib.sha256(source_prepared.to_pdbqt().encode()).hexdigest()
    == original_capture['partner']['sha256']
    == 'dc4e89dc86bd02012b6e14ed1432fc470c1366ba2f66345fe27dc818ce2b873b'
)
assert receptor_path.read_bytes() == base64.b64decode(
    original_capture['receptor']['content_base64'], validate=True
)
assert (
    hashlib.sha256((previous_attempt / 'preparation_1.json').read_bytes()).hexdigest()
    == '091b54c51c5c7ec44b9f3f97602e923e27e72202089931343de55ab1a934789d'
)
prior_preparation = json.loads((previous_attempt / 'preparation_1.json').read_text())
for field in ['ids', 'names', 'types']:
    assert source_before['native'][field] == prior_preparation['native'][field]
for field in ['state_id', 'pdbqt_atom_indices', 'torsion_dof']:
    assert (
        source_before['declaration'][field] == prior_preparation['declaration'][field]
    )
np.testing.assert_allclose(
    source_before['native']['charges_e'],
    prior_preparation['native']['charges_e'],
    rtol=0,
    atol=1e-12,
)
np.testing.assert_allclose(
    source_before['coordinates_nm'],
    prior_preparation['coordinates_nm'],
    rtol=0,
    atol=1e-12,
)
witness(
    source_prepared.source_molsys,
    source_prepared.source_molsys,
    'reproduced_source13_witness',
)
save(
    output / 'reproduced_source_binding.json',
    dict(
        existing_result_sha256=hashlib.sha256(previous_bytes).hexdigest(),
        reproduced_prepared_pdbqt_sha256=original_capture['partner']['sha256'],
        scientific_fields_match=True,
        scope='Source reproduced through documented public preparation; inspection sidecars compared only, never restored.',
    ),
)
source_receptor = receptor_path
source_domain = dmt.BoxRegion(
    puw.quantity(contract['backend_box']['center'], 'angstrom'),
    puw.quantity([24, 24, 24], 'angstrom'),
)
problem = dmt.DockingProblem(
    receptor=source_receptor,
    partner=source_prepared,
    search_domain=source_domain,
    metadata=dict(
        receptor_state_id='e1-captured-3PTB-receptor-20261020',
        partner_state_id=source_prepared.state_id,
    ),
)
save(output / 'source_problem_declaration.json', problem.to_dict())
assert len(source_result) == 1
source_result_before = source_result.to_dict()
source_audit = dmt.audit_result(source_result_before)
save(output / 'source_audit.json', source_audit)
assert source_audit['status'] == 'consistent'
assert (
    source_result.ranking_history
    and source_result.ranking_history[0]['origin'] == 'backend'
)
assert all(len(p.scores) == 4 and p.rank == i for i, p in enumerate(source_result, 1))
assert puw.configure.report() == active_policy
active_torsion_selections = ["atom_name in ['C3','C1']"]
output_directory = output / 'user'
user_notebook = Path(os.environ['DOCKINGMT_E1_USER_NOTEBOOK'])
assert (
    hashlib.sha256(user_notebook.read_bytes()).hexdigest()
    == os.environ['DOCKINGMT_E1_USER_NOTEBOOK_SHA256']
)
user_namespace = dict(
    __name__='__main__',
    source_result=source_result,
    source_prepared=source_prepared,
    source_receptor=source_receptor,
    source_domain=source_domain,
    active_torsion_selections=active_torsion_selections,
    output_directory=output_directory,
)
for cell in json.loads(user_notebook.read_text())['cells']:
    if cell['cell_type'] == 'code':
        exec(
            compile(
                ''.join(cell['source']), 'public_user_example.' + cell['id'], 'exec'
            ),
            user_namespace,
        )
rescored_result = user_namespace['rescored_result']
prepared_problems = user_namespace['prepared_problems']

In [ ]:
assert counts['prepare'] == 2 and counts['reconstruct'] == 1
assert counts['score_returns'] == 2 and counts['score_attempts'] == 2
assert inspect_prepared(source_prepared) == source_before
assert source_result.to_dict() == source_result_before
assert inspect_native(full18, mechanics=False) == full18_before
assert len(prepared_problems) == len(reconstructions) == 1
for index, (original_pose, prepared_problem, (posed, posed_before)) in enumerate(
    zip(source_result, prepared_problems, reconstructions), 1
):
    prepared = prepared_problem.partner
    assert inspect_native(posed) == posed_before
    assert (
        prepared.state_id == source_prepared.state_id
        and prepared.n_atoms == 13
        and prepared.torsion_dof == 1
    )
    witness(posed, source_prepared.source_molsys, f'pose_{index}_reconstructed_witness')
    witness(prepared.source_molsys, posed, f'pose_{index}_prepared_witness')
    charge_audit(prepared, f'pose_{index}_charge_audit')
    np.testing.assert_allclose(
        prepared.charges, source_prepared.charges, rtol=0, atol=1e-12
    )
    assert prepared.atom_types == source_prepared.atom_types
    ids = list(msm.get(posed, element='atom', atom_id=True))
    assert ids == source_before['native']['ids']
    np.testing.assert_allclose(
        puw.get_value(prepared.coordinates, to_unit='nm'),
        puw.get_value(
            msm.get(posed, element='atom', coordinates=True)[0], to_unit='nm'
        ),
        rtol=0,
        atol=1e-12,
    )
    xyz = puw.get_value(prepared.coordinates, to_unit='nm')[prepared.pdbqt_atom_indices]
    np.testing.assert_allclose(
        xyz, puw.get_value(original_pose.coordinates, to_unit='nm'), rtol=0, atol=1e-12
    )
    rendered = export_fields(prepared)
    for key in ['names', 'types', 'tree']:
        assert rendered[key] == source_export[key], key
    np.testing.assert_allclose(
        rendered['charges_e'], source_export['charges_e'], rtol=0, atol=1e-12
    )
    save(output / f'pose_{index}_export_fields.json', rendered)
for index, (old, current) in enumerate(zip(source_result, rescored_result), 1):
    old_record, current_record = old.to_dict(), current.to_dict()
    assert len(current.scores) == 20 and len(current.metadata['scoring_history']) == 2
    assert current_record['coordinates'] == old_record['coordinates']
    for field in ['rank', 'pose_id', 'partner_state_id', 'receptor_state_id']:
        assert current_record[field] == old_record[field]
    for name, value in old.scores.items():
        assert (
            current.scores[name] == value
            and current.score_definitions[name] == old.score_definitions[name]
        )
    for field, value in old.metadata.items():
        if field != 'score_definitions':
            assert current.metadata[field] == value
    assert json.loads(json.dumps(current_record, allow_nan=False)) == current_record
for number, pose in enumerate(returned_scores, 1):
    record = pose.to_dict()
    audit = dmt.audit_pose(record)
    save(output / f'returned_score_{number}_audit.json', audit)
    assert audit['status'] == 'consistent'
    assert (
        dmt.DockingPose.from_dict(
            json.loads(json.dumps(record, allow_nan=False))
        ).to_dict()
        == record
    )
    evaluation = record['metadata']['scoring_history'][-1]
    dmt.verify_captured_inputs(evaluation['backend_artifacts'])
    assert (
        base64.b64decode(
            evaluation['backend_artifacts']['receptor']['content_base64'], validate=True
        )
        == receptor_path.read_bytes()
    )
    expected_partner = prepared_problems[(number - 1) // 2].partner.to_pdbqt().encode()
    assert (
        base64.b64decode(
            evaluation['backend_artifacts']['partner']['content_base64'], validate=True
        )
        == expected_partner
    )
assert rescored_result.ranking_history == source_result.ranking_history
updated_record = rescored_result.to_dict()
for field in ['provenance', 'protocol_info', 'problem_info']:
    assert updated_record[field] == source_result_before[field]
updated_audit = dmt.audit_result(updated_record)
save(output / 'combined_audit.json', updated_audit)
assert updated_audit['status'] == 'consistent'
changed = deepcopy(source_result_before['poses'][0])
changed['partner_state_id'] = 'other-declared-state'
changed_pose = dmt.DockingPose.from_dict(changed)
controls = [
    (
        'changed_state',
        prepared_problems[0],
        changed_pose,
        'partner state does not match',
    )
]
refusals = []
for name, bad_problem, bad_pose, expected in controls:
    before = bad_pose.to_dict()
    try:
        dmt.score(
            bad_problem,
            dmt.VinaProtocol(scoring='vina', cpu=1, seed=7),
            pose=bad_pose,
            score_name='control_' + name,
        )
    except CatalogException as error:
        assert expected in str(error), str(error)
        consume('expected_score_refusals')
        refusals.append(
            dict(control=name, code=getattr(error, 'code', None), message=str(error))
        )
    else:
        raise AssertionError('Expected scoring refusal: ' + name)
    assert bad_pose.to_dict() == before
save(output / 'scoring_refusals.json', refusals)
save(
    output / 'comparison_scope.json',
    dict(
        source_poses=1,
        geometry_swap_control='not_received_single_pose',
        component_rank_refusals='not_applicable_to_singleton',
        positive_comparison_policy_received=False,
    ),
)
assert rescored_result.to_dict() == updated_record
assert source_result.to_dict() == source_result_before
assert counts == budgets
assert puw.configure.report() == active_policy
unit_context.__exit__(None, None, None)
assert puw.configure.report() == caller_policy
assert engine_payload() == engine_before
assert {
    p.name: digest_file(p) for p in (shared / 'conda-meta').glob('*.json')
} == shared_metadata_before
save(
    output / 'native_identity.json',
    dict(
        engine_payload=engine_before,
        libraries_before=initial_libraries,
        libraries_after=native_libraries(),
        versions={
            name: metadata.version(name)
            for name in [
                'dockingmt',
                'molsysmt',
                'vina',
                'argdigest',
                'pyunitwizard',
                'depdigest',
                'smonitor',
            ]
        },
        scope='Exact private installed consumer, borrowed provider/native dependency composition; no independent kernel accuracy or source/build correspondence.',
    ),
)
installed_check('after-public-composition')
save(
    output / 'native_receipt.json',
    dict(
        schema='dockingmt.e1_public_composition_receiving@1',
        owner='uibcdf/dockingmt#78',
        counts=counts,
        source_poses=1,
        original_score_values=4,
        added_fixed_score_values=16,
        original_ranking_history_retained=True,
        source_graph_and_mechanical_fields_received=True,
        state_refusals=1,
        strict_component_ranking_refusals=0,
        native_searches=0,
        successful_fixed_score_calls=2,
        full_e1_accepted=False,
        positive_user_comparison_received=False,
        historical_prepared13_identity_received=False,
        scope='Existing single-pose private installed public composition; source scores/ranks/history preserved; no old-population reproduction, affinity or public closure.',
    ),
)
print(
    json.dumps(
        dict(outcome='PASS', counts=counts, original_scores=4, new_fixed_values=16)
    )
)